In [62]:
from langgraph.graph import StateGraph,START,END
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from typing import TypedDict
load_dotenv()

True

In [63]:
model = ChatGroq(
    model="openai/gpt-oss-120b"
)

In [64]:
class LLMState(TypedDict):
    topic : str
    outline : str
    blog : str
    rating : int

In [65]:
def create_outline(state:LLMState) -> LLMState:
    topic = state['topic']

    prompt = f'generate an outline on the given {topic}'

    outline = model.invoke(prompt).content

    state['outline'] = outline

    return state

In [66]:
def create_blog(state:LLMState) -> LLMState:
    topic = state['topic']
    outline = state['outline']

    prompt = f'generate a blog on the topic {topic} using the outline {outline}'

    blog = model.invoke(prompt).content

    state['blog'] = blog

    return state

In [67]:
def evaluate(state:LLMState) -> LLMState:
    blog = state['blog']

    prompt = f'evaluate and generate a rating on blog {blog} on scale of 1 to 10 and out should should be only integer'

    rating = model.invoke(prompt).content

    state['rating'] = rating

    return state

In [68]:
graph = StateGraph(LLMState)


graph.add_node("outline",create_outline)

graph.add_node("blog",create_blog)

graph.add_node("rating",evaluate)

# edges

graph.add_edge(START,"outline")
graph.add_edge("outline","blog")
graph.add_edge("blog","rating")
graph.add_edge("rating",END)


grap = graph.compile()

In [69]:
initial_state = {"topic":"Ai"}

final_state = grap.invoke(initial_state)


print(final_state)


{'topic': 'Ai', 'outline': 'Below is a flexible, high‑level outline you can adapt to any specific AI system, product, or research project you have in mind.  Feel free to insert the name of the AI (e.g., “ChatGuard\u202fAI,” “EcoVision\u202fAI,” “FinSight\u202fAI”) and flesh out each section with the details that are most relevant to your audience—whether that’s investors, technical reviewers, regulators, or end‑users.\n\n---\n\n## 1. Executive Summary  \n   - **Purpose & Vision** – One‑sentence description of what the AI does and why it matters.  \n   - **Key Benefits** – Top 3‑5 value propositions (e.g., cost savings, accuracy boost, new capability).  \n   - **Target Market / Users** – Primary industries, personas, or problem domains.  \n   - **Milestones & Current Status** – Prototype, beta, production, revenue, etc.  \n\n## 2. Problem Statement  \n   - **Domain Context** – Brief background of the industry or scientific field.  \n   - **Pain Points** – Quantified challenges that exis